# 02 DUSt3R（torch.load 安全补丁）

In [ ]:
import sys, subprocess, urllib.request
from pathlib import Path
import torch
assert torch.cuda.is_available()
print('GPU', torch.cuda.get_device_name(0), torch.__version__)
WORK=Path('/kaggle/working'); IMG=WORK/'mv_images'; OUT=WORK/'outputs_dust3r'; CKPT=WORK/'checkpoints'
for p in [IMG,OUT,CKPT]: p.mkdir(exist_ok=True)

def pip(*a):
    r=subprocess.run([sys.executable,'-m','pip','install','-q',*a], capture_output=True, text=True)
    print('pip',a,r.returncode)
    if r.returncode: print((r.stderr or '')[-800:])

pip('roma','einops','trimesh','matplotlib','plyfile','opencv-python-headless')
REPO=WORK/'dust3r'
if not REPO.exists():
    subprocess.run(f'git clone --recursive --depth 1 https://github.com/naver/dust3r.git {REPO}', shell=True)
req=REPO/'requirements.txt'
if req.exists():
    lines=[ln for ln in req.read_text().splitlines() if not ln.strip().lower().startswith('numpy')]
    fr=WORK/'dust3r_req.txt'; fr.write_text('\n'.join(lines))
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(fr)], check=False)

weight=CKPT/'DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth'
url='https://download.europe.naverlabs.com/ComputerVision/DUSt3R/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth'
if not weight.exists() or weight.stat().st_size < 1_000_000:
    print('dl', url); urllib.request.urlretrieve(url, weight)
print('weight MB', round(weight.stat().st_size/1024/1024,1))

# 只补丁一次，避免递归
import torch as _torch
if not getattr(_torch, '_img3d_load_patched', False):
    _real_load = _torch.load
    def _safe_load(*args, **kwargs):
        if 'weights_only' not in kwargs:
            kwargs['weights_only'] = False
        try:
            return _real_load(*args, **kwargs)
        except TypeError:
            kwargs.pop('weights_only', None)
            return _real_load(*args, **kwargs)
    _torch.load = _safe_load
    _torch._img3d_load_patched = True
    print('patched torch.load once')
else:
    print('torch.load already patched')

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
Image.MAX_IMAGE_PIXELS = 40_000_000
IMG=Path('/kaggle/working/mv_images')
found=[]
for p in Path('/kaggle/working').rglob('*'):
    if not p.is_file(): continue
    if p.suffix.lower() not in {'.png','.jpg','.jpeg'}: continue
    if p.stat().st_size > 12*1024*1024: continue
    if any(k in str(p).lower() for k in ['multiview','set_001','hero_000','gen_images','view_']):
        found.append(p)
seen=set(); uniq=[]
for p in found:
    if p.name in seen: continue
    seen.add(p.name); uniq.append(p)
found=uniq
if len(found)<2:
    for i,col in enumerate([(220,100,80),(100,180,220),(120,200,100),(200,180,60)]):
        im=Image.new('RGB',(512,512),(245,245,245)); d=ImageDraw.Draw(im)
        d.rectangle((150+i*8,150,350+i*8,350), fill=col, outline=(0,0,0), width=3)
        fp=IMG/f'synth_{i}.png'; im.save(fp); found.append(fp)
clean=[]
for i,p in enumerate(found[:6]):
    dst=IMG/f'view_{i:02d}.png'
    im=Image.open(p).convert('RGB'); im.thumbnail((512,512)); im.save(dst); clean.append(dst)
print('views', clean)
assert len(clean)>=2

In [ ]:
import sys
from pathlib import Path
import torch, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shutil

# 保证 patch 存在且不递归
if not getattr(torch, '_img3d_load_patched', False):
    _real_load = torch.load
    def _safe_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        try:
            return _real_load(*args, **kwargs)
        except TypeError:
            kwargs.pop('weights_only', None)
            return _real_load(*args, **kwargs)
    torch.load = _safe_load
    torch._img3d_load_patched = True

REPO=Path('/kaggle/working/dust3r')
sys.path.insert(0,str(REPO)); sys.path.insert(0,str(REPO/'croco'))
OUT=Path('/kaggle/working/outputs_dust3r'); IMG=Path('/kaggle/working/mv_images')
ckpt=str(Path('/kaggle/working/checkpoints/DUSt3R_ViTLarge_BaseDecoder_512_dpt.pth'))

from dust3r.inference import inference
from dust3r.model import AsymmetricCroCo3DStereo
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode

device='cuda'
model=AsymmetricCroCo3DStereo.from_pretrained(ckpt).to(device).eval()
paths=[str(p) for p in sorted(IMG.glob('view_*.png'))]
imgs=load_images(paths, size=512)
pairs=make_pairs(imgs, scene_graph='complete', prefilter=None, symmetrize=True)
output=inference(pairs, model, device, batch_size=1)
scene=global_aligner(output, device=device, mode=GlobalAlignerMode.PointCloudOptimizer)
loss=scene.compute_global_alignment(init='mst', niter=200, schedule='cosine', lr=0.01)
print('align loss', loss)
pts_list=scene.get_pts3d(); conf_list=scene.get_masks()
xyz=[]; rgb=[]
for i,pts in enumerate(pts_list):
    conf=conf_list[i]
    im=imgs[i]['img'].detach().cpu()
    if im.ndim==3 and im.shape[0] in (1,3): im=im.permute(1,2,0)
    im=im.numpy()
    if im.min()<0: im=(im+1)/2
    im=(im*255).clip(0,255).astype(np.uint8)
    pts_np=pts.detach().cpu().numpy()
    conf_np=conf.detach().cpu().numpy() if hasattr(conf,'detach') else np.asarray(conf)
    m=conf_np>0.5 if conf_np.dtype!=bool else conf_np
    if m.shape[:2]!=pts_np.shape[:2]: m=np.ones(pts_np.shape[:2],bool)
    xyz.append(pts_np[m]); rgb.append(im[m] if im.shape[:2]==pts_np.shape[:2] else np.full((int(m.sum()),3),180,np.uint8))
xyz=np.concatenate(xyz,0); rgb=np.concatenate(rgb,0)
print('points', xyz.shape)
n=min(len(xyz),250000)
idx=np.random.choice(len(xyz),n,replace=False) if len(xyz)>n else np.arange(len(xyz))
xyz_s,rgb_s=xyz[idx],rgb[idx]
ply=OUT/'scene_pointcloud.ply'
with open(ply,'w') as f:
    f.write('ply\nformat ascii 1.0\n')
    f.write(f'element vertex {len(xyz_s)}\nproperty float x\nproperty float y\nproperty float z\n')
    f.write('property uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n')
    for p,c in zip(xyz_s,rgb_s):
        f.write(f'{p[0]} {p[1]} {p[2]} {int(c[0])} {int(c[1])} {int(c[2])}\n')
print('saved', ply, ply.stat().st_size)
try:
    fig=plt.figure(figsize=(5,5)); ax=fig.add_subplot(111,projection='3d')
    step=max(len(xyz_s)//5000,1)
    ax.scatter(xyz_s[::step,0],xyz_s[::step,1],xyz_s[::step,2],s=1,c=rgb_s[::step]/255.0)
    fig.savefig(OUT/'preview.png'); plt.close(fig)
except Exception as e:
    print('preview skip', e)
shutil.make_archive('/kaggle/working/dust3r_export','zip', OUT)
print('02 DONE')
assert ply.stat().st_size>1000